In [3]:
try:
    import pandas_ta_classic as ta
    print("✅ Success! Imported as 'pandas_ta_classic'")
except ImportError:
    print("❌ Could not import 'pandas_ta_classic' either.")

✅ Success! Imported as 'pandas_ta_classic'


In [ ]:
import os
import json
import time
import smtplib
import schedule
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta_classic as ta
import gspread
import telegram
from email.message import EmailMessage
from oauth2client.service_account import ServiceAccountCredentials
from dotenv import load_dotenv
from sklearn.metrics import precision_score # For internal tracking if needed

# --- 0. INITIAL SETUP & SECURITY ---
load_dotenv() # Loads secrets from .env

class CryptoBot:
    def __init__(self):
        # Sensitive Credentials (from .env)
        self.TELEGRAM_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN')
        self.CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')
        self.GMAIL_USER = os.getenv('GMAIL_ADDRESS')
        self.GMAIL_PASS = os.getenv('GMAIL_APP_PASSWORD')
        self.SERVICE_ACCOUNT = os.getenv('GCP_SERVICE_ACCOUNT_FILE')
        
        # Configuration Management
        self.CONFIG_CACHE_FILE = "config_cache.json"
        self.config = self.load_config()
        
        # Portfolio State (Simplified for 24/7 run)
        self.cash_usd = self.config.get('BUDGET_USD', 1000)
        self.btc_balance = 0.0
        self.last_dca_price = None
        self.active_trades = [] # Track swing trades for ATR stop-loss
        self.trade_history = []
        
        # Initialize Bot
        if self.TELEGRAM_TOKEN:
            self.bot = telegram.Bot(token=self.TELEGRAM_TOKEN)
        print("Smart ₿itcoin Trading System Initialized.")

    # --- 1. CONFIGURATION MANAGEMENT ---
    def load_config(self):
        """Pulls from Google Sheet, falls back to JSON cache."""
        try:
            # Setup GSheet connection
            scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
            creds = ServiceAccountCredentials.from_json_keyfile_name(self.SERVICE_ACCOUNT, scope)
            client = gspread.authorize(creds)
            sheet = client.open("CryptoBot_Config").sheet1
            
            # Read all rows as a dictionary
            data = sheet.get_all_records()
            new_config = {row['Key']: row['Value'] for row in data}
            
            # Cache locally
            with open(self.CONFIG_CACHE_FILE, 'w') as f:
                json.dump(new_config, f)
            return new_config
        except Exception as e:
            print(f"⚠️ GSheet Error, using cache: {e}")
            with open(self.CONFIG_CACHE_FILE, 'r') as f:
                return json.load(f)

    # --- 2. DATA & FEATURE ENGINEERING ---
    def fetch_and_engineer(self):
        """Scrapes data and calculates TA indicators like ATR and RSI."""
        # Fetching last 7 days of 1-hour data for ATR calculation
        df = yf.download("BTC-USD", period="7d", interval="1h", progress=False)
        
        # Indicators using pandas-ta
        df.ta.atr(length=14, append=True)
        df.ta.rsi(length=14, append=True)
        df.ta.sma(length=20, append=True)
        
        return df

    # --- 3. THE HYBRID STRATEGY ENGINE ---
    def execute_logic(self):
        data = self.fetch_and_engineer()
        current_price = data['Close'].iloc[-1]
        current_atr = data['ATRr_14'].iloc[-1]
        
        # A. ATR-BASED STOP LOSS CHECK
        self.check_stop_losses(current_price, current_atr)
        
        # B. DCA STRATEGY (The Base Layer)
        dca_trigger = float(self.config.get('DCA_DROP_PERCENT', 0.03))
        if self.last_dca_price is None or (self.last_dca_price - current_price) / self.last_dca_price >= dca_trigger:
            self.buy_btc(float(self.config.get('DCA_AMOUNT', 500)), current_price, "DCA Buy")
            self.last_dca_price = current_price

        # C. LLM-ASSISTED OPPORTUNISTIC TRADE
        # Mocking LLM Suggestion based on RSI context
        rsi = data['RSI_14'].iloc[-1]
        if rsi < 30: # Oversold
            self.buy_btc(200, current_price, "LLM Suggestion: Oversold RSI Opportunistic Buy")

    def buy_btc(self, usd_amount, price, reason):
        if self.cash_usd >= usd_amount:
            btc_bought = usd_amount / price
            self.cash_usd -= usd_amount
            self.btc_balance += btc_bought
            msg = f"✅ {reason}: Bought {btc_bought:.6f} BTC at ${price:,.2f}"
            self.notify_trade(msg)
            # Add to active trades for stop-loss tracking
            self.active_trades.append({'entry_price': price, 'amount': btc_bought})
            self.trade_history.append(msg)

    def check_stop_losses(self, current_price, atr):
        k = float(self.config.get('ATR_K_MULTIPLIER', 1.5))
        for trade in self.active_trades[:]:
            stop_price = trade['entry_price'] - (k * atr)
            if current_price <= stop_price:
                self.btc_balance -= trade['amount']
                self.cash_usd += (trade['amount'] * current_price)
                msg = f"🛑 Stop-Loss Hit: Sold at ${current_price:,.2f} (Stop was ${stop_price:,.2f})"
                self.notify_trade(msg)
                self.active_trades.remove(trade)
                self.trade_history.append(msg)

    # --- MONITORING & REPORTING ---
    def notify_trade(self, message):
        print(message)
        if self.TELEGRAM_TOKEN:
            try:
                self.bot.send_message(chat_id=self.CHAT_ID, text=message)
            except Exception as e: print(f"Telegram fail: {e}")

    def send_weekly_email(self):
        msg = EmailMessage()
        portfolio_val = self.cash_usd + (self.btc_balance * yf.Ticker("BTC-USD").fast_info['lastPrice'])
        content = f"Weekly Report\nPortfolio Value: ${portfolio_val:,.2f}\nTrades this week: {len(self.trade_history)}"
        msg.set_content(content)
        msg['Subject'] = "MonReader Crypto: Weekly Trading Summary"
        msg['From'] = self.GMAIL_USER
        msg['To'] = self.GMAIL_USER # Sends to yourself
        
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(self.GMAIL_USER, self.GMAIL_PASS)
            smtp.send_message(msg)
        print("📧 Weekly Report Sent.")
        self.trade_history = [] # Reset for next week

# --- MAIN EXECUTION LOOP ---
if __name__ == "__main__":
    agent = CryptoBot()
    
    # Schedules the weekly report
    schedule.every().monday.at("09:00").do(agent.send_weekly_email)
    
    # Main 24/7 Loop
    while True:
        agent.config = agent.load_config() # Hourly refresh happens here logically
        agent.execute_logic()
        schedule.run_pending()
        time.sleep(1800) # Run every 30 minutes